# IBM Telco Customer Churn: Data Analytics & Feature Pipeline
### Hands-on Practical Pipeline (Stage 1 to Stage 4)
**Target Dataset:** `WA_Fn-UseC_-Telco-Customer-Churn.csv` (7,043 rows, 21 columns)

---
### Notebook Objectives & Structure:
Each stage and individual operation is cleanly separated into its own dedicated cell with explanatory markdown:

- **Stage 1: Problem Definition & Scoping**
  - Business Objective & Loss Mapping (Cost of False Negatives vs False Positives)
  - Success Criteria (Recall $\ge 0.78$, ROC-AUC $\ge 0.84$) & GDPR Compliance
- **Stage 2: Data Ingestion & Low-Level Hygiene**
  - Step 1: File Integrity & Zero-Byte Verification (`Path.stat().st_size`, `hashlib.sha256()`)
  - Step 2: Direct Dataset Ingestion (`pd.read_csv()`)
  - Step 3: Initial Memory Footprint Inspection
  - Step 4: Practical Memory Optimization via Type Downcasting (`int64` $	o$ `int8`/`int16`, `float64` $	o$ `float32`)
  - Step 5: Out-of-Core Chunked Streaming Demonstration
  - Step 6: Export Stage 2 Ingestion CSV (`stage2_ingested_telco.csv`)
- **Stage 3: Data Preprocessing, Cleaning & EDA ("Dirty Data Audit")**
  - Step 1: Schema Standardization (`snake_case` conversion)
  - Step 2: String Normalization & Whitespace Trimming
  - Step 3: Missing Mechanism Analysis (**MCAR, MAR, MNAR**) & whitespace audit in `total_charges`
  - Step 4: Missingness Indicator Flagging (`total_charges_is_missing = 1`)
  - Step 5: Domain & Automated Imputation (`fillna(0.0)`, `SimpleImputer` Median / Mode)
  - Step 6: Deduplication & Target Encoding (`churn`: Yes $	o 1$, No $	o 0$)
  - Step 7: Outlier Detection & Capping (Tukey's IQR Fence Method)
  - Step 8: Exploratory Data Analysis (Summary statistics, correlations, segment churn rates)
  - Step 9: Export Stage 3 Cleaned CSV (`stage3_cleaned_telco.csv`)
- **Stage 4: Feature Engineering, Transformation & Selection**
  - Step 1: PII Removal (`customer_id`) & Strict Stratified Train/Test Split (Fit-Transform Separation)
  - Step 2: Domain Feature Engineering (`charge_per_tenure`, `total_to_monthly_ratio`, `active_services_count`, `has_fiber_no_techsupport`, `contract_loyalty_score`)
  - Step 3: Cardinality Reduction (Grouping rare categories $< 2\%$ to `'Other'`)
  - Step 4: Categorical Encoding (Binary Mapping & `OneHotEncoder(drop='first')`)
  - Step 5: Numerical Feature Scaling (`StandardScaler`)
  - Step 6: Multistage Feature Selection (Zero-Variance, Quasi-Constant, Correlation Filter, Mutual Information `SelectKBest`)
  - Step 7: Dimensionality Reduction via Unsupervised Projection (`PCA(n_components=0.95)`)
  - Step 8: Artifact Serialization (Persisting fitted transformers with `joblib`)
  - Step 9: Export Stage 4 Feature Deliverables (`train_features.csv` & `test_features.csv`)

## 0. Setup & Library Imports
Importing all required standard libraries for file inspection, preprocessing, feature engineering, selection, and serialization.

In [3]:
# =============================================================================
# 0. SETUP & LIBRARY IMPORTS (PRACTICE)
# =============================================================================
import re
import hashlib
import pandas as pd
import numpy as np
import joblib
# from pathlib import path
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold,SelectKBest,mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.decomposition import PCA


---
## Stage 1: Problem Definition & Scoping

### 1. Business Objective Translation
- **Business Problem:** Telecom customer acquisition costs ($300 - $600) drastically exceed retention costs ($20 - $50). Subscriber churn directly damages recurring annual revenue (ARR).
- **ML Formulation:** Supervised Binary Classification (`churn`: 1 = Churned, 0 = Retained).
- **Cost of Errors:**
  - *False Negative (FN):* High cost. A churner is missed and leaves ($500+ lifetime revenue lost).
  - *False Positive (FP):* Low cost. A non-churner receives an unnecessary retention offer ($25 discount cost).
  - *Decision Metric:* Prioritize **Recall $\ge 0.78$** (catch maximum churners) and **ROC-AUC $\ge 0.84$** with well-calibrated probabilities.

### 2. Constraints & Governance
- **GDPR Compliance:** `customerID` is unique pseudonymous personal identifier (PII) and must be dropped prior to modeling.
- **Operational Mode:** Batch scoring monthly prior to customer billing/renewal cycles.

In [4]:
# =============================================================================
# STAGE 1: BUSINESS SCOPING & TARGET VERIFICATION (PRACTICE)
# =============================================================================
# TODO 1: Load raw dataset into 'df_raw_check' from 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
data = pd.read_csv(r"D:\Engineering\CDAC_ALL_MODULES\ML\Parth_DA_ML(All Stages)\Telco-customer\WA_Fn-UseC_-Telco-Customer-Churn.csv")




# TODO 2: Calculate baseline cohort metrics:
#         - total_records = len(df_raw_check)
#         - churn_yes = count where df_raw_check['Churn'] == 'Yes'
#         - churn_no = count where df_raw_check['Churn'] == 'No'
#         - baseline_churn_rate = (churn_yes / total_records) * 100
#
# TODO 3: Print a clean business summary:
#         - Total Cohort Size
#         - Retained Count (No) vs Churned Count (Yes)
#         - Baseline Churn Rate percentage
#         - Class Imbalance ratio (approx 73% vs 27%)



---
## Stage 2: Data Ingestion & Low-Level Hygiene

**Goal:** Ensure file integrity before touching data, load the dataset cleanly, inspect initial memory usage, aggressively downcast numerical data types (`int64` $	o$ `int8`/`int16`, `float64` $	o$ `float32`) without losing precision to minimize RAM footprint, demonstrate out-of-core chunked streaming, and save the ingested file.

### Step 1: File Integrity & Zero-Byte Verification
- **Zero-Byte Check:** Verifies file size $> 0$ bytes using `pathlib.Path.stat().st_size`.
- **SHA-256 Checksum:** Computes cryptographic hash using `hashlib.sha256()` to ensure payloads are untampered.

### Step 2: Direct Dataset Ingestion
Loading the flat file into memory and verifying initial shape.

In [6]:
# =============================================================================
# STAGE 2 - STEP 2: DIRECT DATASET INGESTION (PRACTICE)
# =============================================================================
# TODO 1: Ingest the raw CSV into DataFrame 'df' using pd.read_csv(dataset_path)
df = pd.read_csv(r"D:\Engineering\CDAC_ALL_MODULES\ML\Parth_DA_ML(All Stages)\Telco-customer\WA_Fn-UseC_-Telco-Customer-Churn.csv")

# TODO 2: Inspect raw dimensions: print df.shape
df.shape

# TODO 3: Inspect sample observations: display df.head(5)
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Step 3: Initial Memory Footprint Inspection
Checking baseline column types and deep RAM consumption.

In [8]:
# =============================================================================
# STAGE 2 - STEP 3: INITIAL MEMORY FOOTPRINT INSPECTION (PRACTICE)
# =============================================================================
# TODO 1: Inspect column data types and non-null counts using df.info()
df.info()
# TODO 2: Calculate initial in-memory footprint in Megabytes:
#         initial_memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
df.dtypes
# TODO 3: Print initial memory footprint (MB) formatted to 2 decimal places



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

### Step 4: Practical Memory Optimization (Type Downcasting)
Aggressively downcasting `int64` and `float64` to minimal representations (`int8`, `int16`, `float32`) without precision loss.

In [9]:
# =============================================================================
# STAGE 2 - STEP 4: PRACTICAL MEMORY OPTIMIZATION (TYPE DOWNCASTING) (PRACTICE)
# =============================================================================
# TODO 1: Downcast integer columns (int64 -> smaller signed integer):
#         - Loop over df.select_dtypes(include=['int64']).columns
#         - Apply: pd.to_numeric(df[col], downcast='integer')
for col in df.select_dtypes(include=["int64"]).columns:
    df[col] = pd.to_numeric(df[col],downcast='integer')

# TODO 2: Downcast float columns (float64 -> float32):
#         - Loop over df.select_dtypes(include=['float64']).columns
#         - Apply: pd.to_numeric(df[col], downcast='float')
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = pd.to_numeric(df[col],downcast='float')

print(df.dtypes)


customerID           object
gender               object
SeniorCitizen          int8
Partner              object
Dependents           object
tenure                 int8
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float32
TotalCharges         object
Churn                object
dtype: object


### Step 5: Out-of-Core Processing Demonstration
Demonstrates streaming large datasets in bounded chunks (`chunksize=2000`) to avoid out-of-memory crashes.

### Step 6: Export Stage 2 Ingestion CSV
Saving the memory-optimized ingested dataset as a pipeline checkpoint.

---
## Stage 3: Data Preprocessing, Cleaning & EDA ("Dirty Data Audit")

**Goal:** Clean dirty telemetry: standardize column names to `snake_case`, trim strings, diagnose missing value mechanisms (**MCAR, MAR, MNAR**), flag missingness indicators, apply domain & automated imputation, deduplicate, cap outliers using Tukey's IQR fences, and run exploratory data analysis (EDA).

### Step 1: Schema Standardization (`snake_case`)
Converting all column headers to uniform lowercase `snake_case` using regular expressions.

In [ ]:
# =============================================================================
# STAGE 3 - STEP 1: SCHEMA STANDARDIZATION (snake_case) (PRACTICE)
# =============================================================================
# TODO 1: Write helper function to_snake_case(name: str) -> str:
#         - Use re.sub(r'(?<!^)(?=[A-Z])', '_', name).lower()
#         - Replace spaces ' ' and hyphens '-' with '_'
#         - Strip redundant consecutive underscores re.sub(r'_+', '_', s)


# TODO 2: Apply to_snake_case to all DataFrame column names:
#         df.columns = [to_snake_case(col) for col in df.columns]

# TODO 3: Print standardized column list



### Step 2: String Normalization & Whitespace Trimming
Trimming leading/trailing whitespace across all text/object columns.

In [11]:
# =============================================================================
# STAGE 3 - STEP 2: STRING NORMALIZATION & WHITESPACE TRIMMING (PRACTICE)
# =============================================================================
# TODO 1: Loop over all object columns: df.select_dtypes(include=['object']).columns
for col in df.select_dtypes(include=['object']).columns:
    if col != 'customer_id':
        df[col] = df[col].astype(str).str.strip()

df.head(5)



,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.850000,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.950001,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.849998,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.299999,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.699997,151.65,Yes


### Step 3: Missing Mechanism Analysis (MCAR / MAR / MNAR)
- **MCAR (Missing Completely at Random):** Missingness is pure random noise; unrelated to any variable.
- **MAR (Missing at Random):** Missingness is systematically explained by another observed feature.
  - *Audit:* `total_charges` contains 11 whitespace strings `" "`. All 11 records have `tenure == 0` (brand-new customers who haven't completed a billing cycle) $	o$ **MAR**.
- **MNAR (Missing Not at Random):** Missingness depends directly on the unobserved value itself (e.g. sentinel codes like -999).

In [ ]:
# =============================================================================
# STAGE 3 - STEP 3: MISSING MECHANISM ANALYSIS (MCAR / MAR / MNAR) (PRACTICE)
# =============================================================================
# TODO 1: Identify blank space ' ' or empty string '' records in 'total_charges':
#         blank_mask = (df['total_charges'] == ' ') | (df['total_charges'] == '')
#         Print count of blank records


# TODO 2: Inspect tenure for these blank records:
#         affected_tenures = df.loc[blank_mask, 'tenure'].unique()
#         Print unique tenure values

# TODO 3: Diagnose missingness mechanism:
#         Confirm whether missing total_charges occur exclusively when tenure == 0 (MAR)



### Step 4: Missingness Indicator Flagging
Preserving the "signal of absence" prior to filling values so downstream ML models can distinguish brand-new subscribers.

In [ ]:
# =============================================================================
# STAGE 3 - STEP 4: MISSINGNESS INDICATOR FLAGGING (PRACTICE)
# =============================================================================
# TODO 1: Preserve the 'signal of absence' by creating a binary flag column:
#         df['total_charges_is_missing'] = blank_mask.astype(int)
#
# TODO 2: Print confirmation and verify count of flagged records (should be 11)



### Step 5: Domain & Automated Imputation (Median / Mode / Multivariate)
- **Domain-Specific Imputation (`fillna(0.0)`):** For customers with `tenure == 0`, accumulated charges are logically `0.0`.
- **`SimpleImputer(strategy='median')`:** Automated median imputation for continuous skewed numerics.
- **`SimpleImputer(strategy='most_frequent')`:** Automated mode imputation for categorical columns.
- *Multivariate Patterns:* `KNNImputer()` and `IterativeImputer()` (MICE) documented for multi-feature regression imputation.

In [ ]:
# =============================================================================
# STAGE 3 - STEP 5: DOMAIN & AUTOMATED IMPUTATION PIPELINE (PRACTICE)
# =============================================================================
# TODO 1: Convert 'total_charges' to numeric, replacing ' ' with np.nan:
#         df['total_charges'] = pd.to_numeric(df['total_charges'].replace(' ', np.nan))
#
# TODO 2: Apply Domain-specific Imputation:
#         For tenure == 0, total_charges accumulated is logically 0.0:
#         df['total_charges'] = df['total_charges'].fillna(0.0)
#
# TODO 3: Setup Automated Sklearn Imputation for potential remaining nulls:
#         - Median Imputation for skewed numeric columns using SimpleImputer(strategy='median')
#         - Mode Imputation for categorical columns using SimpleImputer(strategy='most_frequent')
#
# TODO 4: Verify zero nulls remain across entire DataFrame: assert df.isnull().sum().sum() == 0



### Step 6: Deduplication & Target Mapping
Removing duplicate records and mapping target `churn`: `'Yes'` $	o 1$, `'No'` $	o 0$.

In [ ]:
# =============================================================================
# STAGE 3 - STEP 6: DEDUPLICATION & TARGET MAPPING (PRACTICE)
# =============================================================================
# TODO 1: Audit duplicate records: print df.duplicated().sum()
#         Drop duplicates if any: df.drop_duplicates(inplace=True)
#
# TODO 2: Convert target variable 'churn' into binary integer (1 = Churned, 0 = Retained):
#         churn_mapping = {'Yes': 1, 'No': 0, 'yes': 1, 'no': 0}
#         df['churn'] = df['churn'].map(churn_mapping).astype(int)
#
# TODO 3: Print target value counts and normalized class percentages



### Step 7: Outlier Detection & Capping (Tukey's IQR Fence Method)
Calculating Tukey's fences $[Q_1 - 1.5 	imes IQR, \ Q_3 + 1.5 	imes IQR]$ to cap extreme numerical values without dropping data.

In [ ]:
# =============================================================================
# STAGE 3 - STEP 7: OUTLIER DETECTION & CAPPING (TUKEY'S IQR FENCES) (PRACTICE)
# =============================================================================
# TODO 1: Define continuous columns to audit: ['tenure', 'monthly_charges', 'total_charges']
#
# TODO 2: For each column, calculate:
#         - Q1 = df[col].quantile(0.25)
#         - Q3 = df[col].quantile(0.75)
#         - IQR = Q3 - Q1
#         - lower_fence = Q1 - 1.5 * IQR
#         - upper_fence = Q3 + 1.5 * IQR
#
# TODO 3: Count and print outliers outside [lower_fence, upper_fence]
#
# TODO 4: Apply Tukey Capping (Winsorization) using df[col].clip(lower=..., upper=...)



### Step 8: Exploratory Data Analysis (EDA)
Inspecting continuous feature distributions, correlation with `churn`, and churn rates across service segments.

In [ ]:
# =============================================================================
# STAGE 3 - STEP 8: EXPLORATORY DATA ANALYSIS (EDA) (PRACTICE)
# =============================================================================
# TODO 1: Display 5-number descriptive summary of continuous features:
#         print(df[continuous_cols].describe().round(2))
#
# TODO 2: Compute Pearson correlation of numerical columns against target 'churn':
#         num_corr_cols = continuous_cols + ['senior_citizen', 'total_charges_is_missing', 'churn']
#         print(df[num_corr_cols].corr()['churn'].sort_values(ascending=False).round(4))
#
# TODO 3: Compute Churn Rate grouped by 'contract' type:
#         df.groupby('contract')['churn'].agg(Total='count', Churned='sum', Rate=lambda x: f'{x.mean()*100:.2f}%')
#
# TODO 4: Compute Churn Rate grouped by 'internet_service' type



### Step 9: Export Stage 3 Cleaned CSV
Saving the sanitized dataset as the baseline input for Stage 4.

In [ ]:
# =============================================================================
# STAGE 3 - STEP 9: EXPORT STAGE 3 CLEANED CSV (PRACTICE)
# =============================================================================
# TODO 1: Save the cleaned DataFrame to 'stage3_cleaned_telco.csv' with index=False
#
# TODO 2: Print confirmation of Stage 3 deliverable export



---
## Stage 4: Feature Engineering, Transformation & Selection

**Goal:** Generate domain features, manage cardinality, encode categoricals, scale numerics, select top features, apply PCA dimensionality reduction, and serialize fitted transformer artifacts.

### Step 1: PII Removal & Stratified Train/Test Split
Dropping `customer_id` (PII) and performing an 80/20 stratified split *first* to enforce strict fit-transform separation and prevent data leakage.

In [ ]:
# =============================================================================
# STAGE 4 - STEP 1: PII REMOVAL & STRATIFIED TRAIN/TEST SPLIT (PRACTICE)
# =============================================================================
# TODO 1: Drop PII identifier columns ('customer_id' or 'customerid') robustly:
#         id_cols = [c for c in df.columns if c.lower() in ['customerid', 'customer_id', 'id']]
#         df = df.drop(columns=id_cols)
#
# TODO 2: Separate features X and target y:
#         X = df.drop(columns=['churn'])
#         y = df['churn']
#
# TODO 3: Perform Stratified Train/Test Split (80/20) with random_state=42 and stratify=y:
#         X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
#
# TODO 4: Reset index on all 4 resulting sets (.reset_index(drop=True))
#
# TODO 5: Print shapes and verify identical churn rates across train (y_train.mean()) and test (y_test.mean())



### Step 2: Domain Feature Engineering
Creating high-signal telecom domain features:
- `charge_per_tenure`: monthly spend rate relative to tenure (safe division for `tenure = 0`).
- `total_to_monthly_ratio`: total charges divided by monthly charges (estimated billing cycles).
- `active_services_count`: sum of active digital add-ons.
- `has_fiber_no_techsupport`: interaction flag identifying the highest-risk churn profile.
- `contract_loyalty_score`: ordinal ranking (`Month-to-month` = 1, `One year` = 2, `Two year` = 3).

In [ ]:
# =============================================================================
# STAGE 4 - STEP 2: DOMAIN FEATURE ENGINEERING (PRACTICE)
# =============================================================================
# TODO 1: Create function create_domain_features(data: pd.DataFrame) -> pd.DataFrame:
#         - df_out = data.copy()
#         - Fix column name if needed: rename 'streaming_t_v' -> 'streaming_tv'
#
#         - Feature 1: Ratio of monthly spend to tenure (safe division):
#           df_out['charge_per_tenure'] = df_out['monthly_charges'] / (df_out['tenure'] + 1)
#
#         - Feature 2: Ratio of total charges to monthly charges:
#           df_out['total_to_monthly_ratio'] = df_out['total_charges'] / (df_out['monthly_charges'] + 1e-6)
#
#         - Feature 3: Entertainment streaming bundle flag:
#           df_out['has_streaming'] = ((df_out['streaming_tv'] == 'Yes') | (df_out['streaming_movies'] == 'Yes')).astype(int)
#
#         - Feature 4: Security & Support bundle flag:
#           df_out['has_security_backup'] = ((df_out['online_security'] == 'Yes') & (df_out['online_backup'] == 'Yes')).astype(int)
#
#         - Feature 5: Total subscribed services count (integer sum across 6 service columns):
#           services = ['online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies']
#           df_out['total_services_count'] = (df_out[services] == 'Yes').sum(axis=1)
#
#         - Feature 6: Contract risk flag (Month-to-month contracts have highest churn risk):
#           df_out['is_month_to_month'] = (df_out['contract'] == 'Month-to-month').astype(int)
#
#         - Feature 7: High churn payment method flag ('Electronic check'):
#           df_out['high_risk_electronic_check'] = (df_out['payment_method'] == 'Electronic check').astype(int)
#
#         - Return df_out
#
# TODO 2: Apply create_domain_features() independently to X_train and X_test to prevent leakage:
#         X_train = create_domain_features(X_train)
#         X_test = create_domain_features(X_test)



### Step 3: Cardinality Reduction (Rare Category Grouping)
Grouping infrequent categorical levels (frequency $< 2\%$) into `'Other'` to prevent dimension explosion.

In [ ]:
# =============================================================================
# STAGE 4 - STEP 3: CARDINALITY REDUCTION (RARE CATEGORY GROUPING) (PRACTICE)
# =============================================================================
# TODO 1: Set rare_threshold = 0.02 (categories below 2% frequency are pooled)
#
# TODO 2: Select categorical columns to check: ['payment_method', 'contract', 'internet_service']
#
# TODO 3: For each column:
#         - Compute category frequency on train fold: X_train[col].value_counts(normalize=True)
#         - Identify frequent categories: freq[freq >= rare_threshold].index.tolist()
#         - Pool rare categories on X_train: apply lambda x: x if x in frequent else 'Other'
#         - Pool rare categories on X_test using the SAME train frequent categories (NO LEAKAGE!)
#
# TODO 4: Print confirmation of cardinality reduction



### Step 4: Categorical Encoding (Binary Mapping & One-Hot Encoding)
- Binary mapping for 2-level categories (`partner`, `dependents`, `phone_service`, `paperless_billing`, `gender`).
- Low-cardinality `OneHotEncoder(drop='first', sparse_output=False)` fit strictly on the training fold.

In [ ]:
# =============================================================================
# STAGE 4 - STEP 4: CATEGORICAL ENCODING PIPELINE (PRACTICE)
# =============================================================================
# TODO 1: Binary Mapping for 2-level categories:
#         binary_cols = ['partner', 'dependents', 'phone_service', 'paperless_billing']
#         mapping = {'Yes': 1, 'No': 0, 'yes': 1, 'no': 0}
#         Map on both X_train and X_test
#
# TODO 2: Map gender ('Female' -> 1, 'Male' -> 0) on X_train and X_test
#
# TODO 3: Multi-class One-Hot Encoding:
#         - Identify remaining object columns (multi-class nominal)
#         - Initialize: ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
#         - FIT ohe strictly on X_train[ohe_cols]: ohe.fit(X_train[ohe_cols])
#         - Transform both X_train and X_test
#         - Convert to DataFrames with feature names from ohe.get_feature_names_out(ohe_cols)
#         - Concatenate with non-OHE columns and drop original categorical columns
#
# TODO 4: Verify that all columns in X_train and X_test are strictly numeric



### Step 5: Numerical Feature Scaling (`StandardScaler`)
Standardizing continuous variables to mean 0 and unit variance $\sigma = 1$.

In [ ]:
# =============================================================================
# STAGE 4 - STEP 5: NUMERICAL FEATURE SCALING (StandardScaler) (PRACTICE)
# =============================================================================
# TODO 1: Define continuous numerical features to scale:
#         scale_cols = ['tenure', 'monthly_charges', 'total_charges', 'charge_per_tenure', 'total_to_monthly_ratio']
#
# TODO 2: Initialize StandardScaler(): scaler = StandardScaler()
#
# TODO 3: FIT scaler strictly on X_train[scale_cols]:
#         scaler.fit(X_train[scale_cols])
#
# TODO 4: Transform X_train and X_test:
#         X_train[scale_cols] = scaler.transform(X_train[scale_cols])
#         X_test[scale_cols] = scaler.transform(X_test[scale_cols])
#
# TODO 5: Verify mean approx 0 and std approx 1 on X_train



### Step 6: Multistage Feature Selection
- **Zero-Variance Filter:** Eliminates columns with variance $= 0$.
- **Quasi-Constant Filter:** Drops columns where over $99\%$ of values are identical.
- **Collinearity Filter:** Drops one of any feature pair with Pearson correlation $|r| > 0.85$.
- **Mutual Information `SelectKBest`:** Isolates the top 15 most informative non-linear predictors.

In [ ]:
# =============================================================================
# STAGE 4 - STEP 6: MULTISTAGE FEATURE SELECTION PIPELINE (PRACTICE)
# =============================================================================
# TODO 1: Defensive check: Ensure X_train and X_test contain strictly numeric dtypes
#
# TODO 2: Stage 6A - Variance Filtering (Quasi-Constant Removal):
#         - Initialize VarianceThreshold(threshold=0.01)
#         - Fit on X_train and identify retained feature names
#         - Filter X_train and X_test to non-constant features
#
# TODO 3: Stage 6B - Multicollinearity Filtering (Pearson Correlation):
#         - Compute correlation matrix: corr_matrix = X_train_var.corr().abs()
#         - Identify pairs with correlation > 0.85
#         - Identify and drop redundant features from both X_train and X_test
#
# TODO 4: Stage 6C - Information-Theoretic Selection (Mutual Information):
#         - Initialize selector = SelectKBest(score_func=mutual_info_classif, k=15)
#         - FIT selector strictly on X_train and y_train
#         - Transform X_train and X_test into X_train_selected and X_test_selected
#         - Extract selected feature names: [col for col, mask in zip(cols, selector.get_support()) if mask]
#         - Print ranked feature list with their mutual information scores



### Step 7: Dimensionality Reduction via Unsupervised Projection (`PCA`)
Compressing high-dimensional correlated feature matrices into orthogonal components while preserving **95% cumulative explained variance** using `sklearn.decomposition.PCA`.

In [ ]:
# =============================================================================
# STAGE 4 - STEP 7: DIMENSIONALITY REDUCTION VIA PCA (PRACTICE)
# =============================================================================
# TODO 1: Initialize PCA(n_components=0.95, random_state=42) to retain 95% variance
#
# TODO 2: FIT PCA strictly on X_train_selected and transform X_train and X_test:
#         X_train_pca = pca.fit_transform(X_train_selected)
#         X_test_pca = pca.transform(X_test_selected)
#
# TODO 3: Print compressed dimensions and explained_variance_ratio_ per component



### Step 8: Artifact Serialization (`joblib`)
Persisting fitted transformer objects (`ohe`, `scaler`, `selector`, `pca`, `selected_features`) to disk using `joblib.dump()` so production inference pipelines apply identical parameters without retraining.

In [ ]:
# =============================================================================
# STAGE 4 - STEP 8: ARTIFACT SERIALIZATION (joblib) (PRACTICE)
# =============================================================================
# TODO 1: Create artifacts directory: artifacts_dir = Path('fitted_artifacts'); artifacts_dir.mkdir(exist_ok=True)
#
# TODO 2: Persist all fitted transformer objects using joblib.dump():
#         - 'ohe_encoder.joblib'       -> ohe
#         - 'scaler.joblib'            -> scaler
#         - 'mi_selector.joblib'       -> selector
#         - 'pca.joblib'               -> pca
#         - 'selected_features.joblib' -> selected_features (list of column names)
#
# TODO 3: Print confirmation verifying all serialized artifacts exist on disk



### Step 9: Export Stage 4 Feature Deliverables
Saving the final engineered train and test matrices (`train_features.csv` & `test_features.csv`).

In [ ]:
# =============================================================================
# STAGE 4 - STEP 9: EXPORT STAGE 4 FEATURE DELIVERABLES (PRACTICE)
# =============================================================================
# TODO 1: Construct final train and test deliverable DataFrames by concatenating features with target:
#         train_export = pd.concat([X_train_selected, y_train.rename('churn')], axis=1)
#         test_export = pd.concat([X_test_selected, y_test.rename('churn')], axis=1)
#
# TODO 2: Export to CSV (index=False):
#         train_export.to_csv('train_features.csv', index=False)
#         test_export.to_csv('test_features.csv', index=False)
#
# TODO 3: Print final export confirmation and summary shapes

